# Flows between strata

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/11-flows-between-strata` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

In the flow-types and stratification introductions, the usual workflow is:

- define an unstratified model and its flows;
- stratify the map, which splits those flows across new strata.

Sometimes the flow itself should move people *between* strata — for example
migration from rural to urban. This page shows that pattern with
`TransitionFlow` source/dest selectors, then age-specific rates via
`Overwrite`, and ageing via `TraitChain`.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    EntryFlow,
    ExitFlow,
    FlowModel,
    Overwrite,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TraitChain,
    TransitionFlow,
)
from summer4.flows.rates import Reduce

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("pop",))
location = Property("location", ("urban", "rural"))
age = Property("age", ("0", "20", "40"))


def plot_totals(res, title):
    frame = res["comp"].to_pandas()
    return frame.plot(title=title, labels={"index": "year", "value": "people"})


def wrap_y0(pmap, arr):
    return PropertyData.wrap(pmap, jnp.asarray(arr))


def run(model, y0, params=None, t0=1990.0, t1=2020.0):
    plan = SavePlan(
        requests={"comp": SaveRequest(Compartments())},
        ts=np.linspace(t0, t1, 61),
    )
    p = {} if params is None else params
    return model.compile().run(p, y0, t0=t0, t1=t1, dt=0.1, save=plan, solver="euler")


## Unstratified model

One compartment with crude birth and death. Births replace deaths at a higher
rate so the population grows (`EntryFlow` takes an absolute rate; here births
are `death.sum() * (birth_rate / death_rate)` so the total birth flux is a
fraction of the whole population).


In [ ]:
pmap0 = PropertyMap.from_property(state)
m0 = FlowModel(pmap0)
death0 = m0.add_flow(ExitFlow("death", state["pop"], Param("death")))
m0.add_flow(EntryFlow("birth", state["pop"], death0.sum() * 2.0))  # birth_rate 0.02

y0_flat = wrap_y0(pmap0, np.array([20e6]))
params = {"death": 0.01}
res0 = run(m0, y0_flat, params)
pop = np.asarray(res0["comp"].total().values)
assert float(pop[-1]) > float(pop[0]), "population should grow"

# JIT gate: finite grad of final population through the death Param.
cm0 = m0.compile()
plan0 = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(1990.0, 2000.0, 11),
)


def loss_death(death: jax.Array) -> jax.Array:
    out = cm0.run(
        {"death": death},
        y0_flat,
        t0=1990.0,
        t1=2000.0,
        dt=0.1,
        save=plan0,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(out["comp"].values.data))


val = float(jax.jit(loss_death)(jnp.asarray(0.01)))
grad = float(jax.jit(jax.grad(loss_death))(jnp.asarray(0.01)))
assert np.isfinite(val) and np.isfinite(grad)
plot_totals(res0, "Unstratified population with birth and death")


## Stratified by location

Split into urban / rural. Initial population is 30% urban, 70% rural. Births
still split evenly across destination strata (summer2's default caveat).


In [ ]:
pmap_loc = PropertyMap.from_property(state).stratify(location)
m_loc = FlowModel(pmap_loc)
death_loc = m_loc.add_flow(ExitFlow("death", state["pop"], Param("death")))
m_loc.add_flow(EntryFlow("birth", state["pop"], death_loc.sum() * 2.0))

y0_arr = np.zeros(pmap_loc.size)
y0_arr[pmap_loc.select(location["urban"])] = 20e6 * 0.3
y0_arr[pmap_loc.select(location["rural"])] = 20e6 * 0.7
y0_loc = wrap_y0(pmap_loc, y0_arr)
res_loc = run(m_loc, y0_loc, params)
urban0 = float(np.asarray(res_loc["comp"].select(location["urban"]).total().values)[0])
rural0 = float(np.asarray(res_loc["comp"].select(location["rural"]).total().values)[0])
assert abs(urban0 / rural0 - 0.3 / 0.7) < 1e-6
plot_totals(res_loc, "Urban / rural population (no migration)")


## Migration between strata

Two percent of the rural population moves to urban each year. Source and dest
selectors pin the edge to `rural → urban` on the same disease state.


In [ ]:
m_mig = FlowModel(pmap_loc)
death_m = m_mig.add_flow(ExitFlow("death", state["pop"], Param("death")))
m_mig.add_flow(EntryFlow("birth", state["pop"], death_m.sum() * 2.0))
m_mig.add_flow(
    TransitionFlow("migration", location["rural"], location["urban"], Param("mig"))
)
assert m_mig.compile().edges("migration").n_edges == 1

mig_params = {"death": 0.01, "mig": 0.02}
res_mig = run(m_mig, y0_loc, mig_params)
urban = np.asarray(res_mig["comp"].select(location["urban"]).total().values)
rural = np.asarray(res_mig["comp"].select(location["rural"]).total().values)
assert float(urban[-1]) > float(urban[0]), "urban should grow via migration"
assert float(rural[-1]) < float(rural[0]) * 1.2  # rural grows slower / shrinks share
plot_totals(res_mig, "Urban / rural with rural→urban migration")


## Age-specific migration

People aged 0–19 rarely migrate; 20–39 migrate fastest; 40+ less so. Age bands
are an ordinary `Property` plus a `TraitChain` ageing flow (summer4 has no
`AgeStratification` class — ledger `S6`). Migration rates are `Overwrite`s on
the shared migration flow.


In [ ]:
pmap_age = pmap_loc.stratify(age)
m_age = FlowModel(pmap_age)
death_a = m_age.add_flow(ExitFlow("death", state["pop"], Param("death")))
m_age.add_flow(EntryFlow("birth", state["pop"], death_a.sum() * 2.0))
m_age.add_flow(
    TransitionFlow(
        "migration",
        location["rural"],
        location["urban"],
        0.0,
        adjust=[
            Overwrite(0.0, where=age["0"]),
            Overwrite(0.05, where=age["20"]),
            Overwrite(0.01, where=age["40"]),
        ],
    )
)
m_age.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        1.0,
        pairing=TraitChain(age, (("0", "20"), ("20", "40")), rates=(1.0 / 20.0, 1.0 / 20.0)),
    )
)

y0_arr = np.zeros(pmap_age.size)
age_share = {"0": 0.2, "20": 0.4, "40": 0.4}
for a, share in age_share.items():
    y0_arr[pmap_age.select(location["urban"] & age[a])] = 20e6 * 0.3 * share
    y0_arr[pmap_age.select(location["rural"] & age[a])] = 20e6 * 0.7 * share
y0_age = wrap_y0(pmap_age, y0_arr)

res_age = run(m_age, y0_age, params)
urban_t = res_age["comp"].select(location["urban"]).total()
rural_t = res_age["comp"].select(location["rural"]).total()
frame = pd.DataFrame(
    {
        "urban": np.asarray(urban_t.values).ravel(),
        "rural": np.asarray(rural_t.values).ravel(),
    },
    index=np.asarray(urban_t.times.values),
)
assert float(frame["urban"].iloc[-1]) > float(frame["urban"].iloc[0])
assert m_age.compile().edges("migration").n_edges == 3
frame.plot(title="Urban vs rural (age-specific migration)", labels={"index": "year", "value": "people"})


## Summary

Inter-stratum flows are ordinary `TransitionFlow`s with source/dest selectors
on the stratification property. Ageing is a `TraitChain`; age-specific rates
are flow-owned `Overwrite` / `Multiply` adjustments — not methods on a
stratification object.
